# kaggle image server

text-to-image on the t4s via diffusers (no comfyui). edit `IMAGE_KEY` in the config cell and run top to bottom; re-run just the generate cell for more images.

flux1-dev and ideogram-4 are **non-commercial** licenses and ideogram-4 + flux are **gated**: accept the license on their hf model pages and add an `HF_TOKEN` kaggle secret first.

In [ ]:
# --- setup: pull in the shared modules -----------------------------------

# option a: clone from the repo.
# rm -rf first so re-running this cell doesn't fail on an existing dir.
!rm -rf /kaggle/working/harness_src && git clone https://github.com/Meru143/kaggle-model-server.git /kaggle/working/harness_src
import sys
sys.path.append("/kaggle/working/harness_src")

# option b: attach the .py files as a Kaggle Dataset instead --
# comment out option a above and uncomment this line instead:
# sys.path.append("/kaggle/input/TODO-your-dataset-slug")

!pip install -q huggingface_hub requests

In [ ]:
import os

# re-running setup + this cell picks up freshly-pulled code without a
# kernel restart (python caches imports; plain re-import returns stale code)
import importlib, sys
if "image_models" in sys.modules:
    importlib.reload(sys.modules["image_models"])

# gated models (flux1-dev, ideogram-4): kaggle secrets are NOT env vars --
# export the attached HF_TOKEN secret so huggingface_hub can see it
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    pass  # no secret attached -- fine for z-image-turbo / krea-2-turbo

from image_models import IMAGE_MODELS, install, load, generate

list(IMAGE_MODELS.keys())

## config — change this line to swap models

In [ ]:
IMAGE_KEY = "z-image-turbo"  # <- change this, then re-run the cells below

In [ ]:
install(IMAGE_KEY)   # pip installs this model's exact requirements
pipe = load(IMAGE_KEY)  # fp16 + nf4 transformer + cpu offload for the t4

In [ ]:
path = generate(pipe, "a red fox sitting in fresh snow, golden hour, photorealistic")

from IPython.display import Image
Image(path)

pngs land in `/kaggle/tmp/outputs/` (ephemeral — download what you want to keep, or copy to `/kaggle/working`).

per-call overrides: `generate(pipe, prompt, num_inference_steps=4, height=768, width=768)`. swapping models mid-session: just change `IMAGE_KEY` and re-run install/load — but `del pipe` first if vram is tight.